# 日级特征 v2

**改进说明（基于 Hamidovic 2023 与 Wang 2025 两篇论文）**：

- **P1** STEP 4 后新增 `day_in_cycle`、`day_in_cycle_norm`（周期位置先验）
- **P2** 主要输出使用 sleep 窗口作为下游模型首选输入
- **P3** STEP 5 缺失值改为邻日线性插值（limit=3 天），避免全局均值抹平相位差异
- **P4** STEP 6 rolling z 改为优先使用**周期内早期（前5天）基线**；如早期数据不足则回退长期窗口 K=28
- **P5** STEP 3 后新增腕温/夜间温度**双相转折衍生特征** `wt_shift_7v3`、`temp_shift_7v3`

STEP 1 质量过滤 → STEP 2 时间窗口 → STEP 3 日级聚合 → STEP 3.5 双相转折衍生特征 → STEP 4 合并+周期位置特征 → STEP 5 缺失/mask → STEP 6 周期内基线 z 归一化 → STEP 7 输出。

In [63]:
import pandas as pd
import numpy as np
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "data_process" else cwd
SUB = ROOT / "subdataset" / "2"
OUT = ROOT / "processed_data" / "2"
OUT.mkdir(parents=True, exist_ok=True)

cycle = pd.read_csv(ROOT / "subdataset" / "cycle_clean_2.csv")
# Use (id, study_interval, day_in_study, small_group_key) to uniquely identify a day in a cycle for downstream merge
cycle_days = cycle[["id", "study_interval", "day_in_study", "small_group_key"]].drop_duplicates()
print("cycle_days:", len(cycle_days))

cycle_days: 4825


**时间约定（据 mcPHASES 文档）**：`day_in_study` = 从研究开始的天数索引（日历日）；`timestamp` = 当日时间 `HH:MM:SS`（本地时间）。故 `(id, study_interval, day_in_study, timestamp)` 表示该日历日该时刻的一条记录。睡眠边界来自 computed_temperature 的 `sleep_start/end_day_in_study` + `sleep_start/end_timestamp`，一夜归属到 **sleep_end_day_in_study**（醒来日）。

In [64]:
def ts_to_min(ts):
    """'HH:MM:SS' -> minutes since midnight (0~1440)."""
    if pd.isna(ts):
        return np.nan
    p = str(ts).strip().split(":")
    h = int(p[0]) if len(p) > 0 else 0
    m = int(float(p[1])) if len(p) > 1 else 0
    s = float(p[2]) if len(p) > 2 else 0
    return h * 60 + m + s / 60

# Sleep bounds: one row per night attributed to sleep_end_day_in_study; rename to day_in_study to avoid duplicate column
ct = pd.read_csv(SUB / "computed_temperature_cycle.csv")
ct_night = ct.groupby(["id", "study_interval", "sleep_end_day_in_study"], as_index=False).first()
ct_night["sleep_start_min"] = ct_night["sleep_start_timestamp"].map(ts_to_min)
ct_night["sleep_end_min"] = ct_night["sleep_end_timestamp"].map(ts_to_min)
sleep_bounds = ct_night[["id", "study_interval", "sleep_end_day_in_study", "sleep_start_min", "sleep_end_min"]].rename(
    columns={"sleep_end_day_in_study": "day_in_study"}
)
print("sleep_bounds rows:", len(sleep_bounds))

sleep_bounds rows: 4437


In [65]:
# STEP 1: HRV quality filter
hrv = pd.read_csv(SUB / "heart_rate_variability_details_cycle.csv")
n_hrv = len(hrv)
hrv = hrv[(hrv["coverage"] >= 0.6) & (hrv["rmssd"] > 0) & (hrv["low_frequency"] > 0) & (hrv["high_frequency"] > 0)]
hrv["timestamp_min"] = hrv["timestamp"].map(ts_to_min)
print("HRV: %.2f%% kept" % (100 * len(hrv) / n_hrv))

HRV: 99.94% kept


In [66]:
def agg_hrv(df):
    g = df.groupby(["id", "study_interval", "day_in_study"], as_index=False)
    lf = g["low_frequency"].mean()
    hf = g["high_frequency"].mean()
    out = g.agg(rmssd_mean=("rmssd", "mean")).merge(lf.rename(columns={"low_frequency": "lf_mean"}))
    out = out.merge(hf.rename(columns={"high_frequency": "hf_mean"}))
    out["lf_hf_ratio"] = out["lf_mean"] / (out["hf_mean"].replace(0, np.nan))
    return out[["id", "study_interval", "day_in_study", "rmssd_mean", "lf_mean", "hf_mean", "lf_hf_ratio"]]

hrv_merged = hrv.merge(sleep_bounds, on=["id", "study_interval", "day_in_study"], how="left")
hrv_full = agg_hrv(hrv)
# sleep: same-day time<=sleep_end + prev-day time>=sleep_start -> attribute to wake_day
hrv_same = hrv_merged[hrv_merged["timestamp_min"] <= hrv_merged["sleep_end_min"]][hrv.columns]
hrv_prev = hrv.copy()
hrv_prev["wake_day"] = hrv_prev["day_in_study"] + 1
hrv_prev = hrv_prev.merge(sleep_bounds.rename(columns={"day_in_study": "wake_day"})[["id", "study_interval", "wake_day", "sleep_start_min"]], on=["id", "study_interval", "wake_day"], how="inner")
hrv_prev = hrv_prev[hrv_prev["timestamp_min"] >= hrv_prev["sleep_start_min"]].drop(columns=["sleep_start_min"], errors="ignore")
hrv_prev["day_in_study"] = hrv_prev["wake_day"]
hrv_prev = hrv_prev[hrv.columns]
hrv_sleep = agg_hrv(pd.concat([hrv_same, hrv_prev], ignore_index=True))
# morning: same-day [sleep_end, sleep_end+30]
hrv_morning = agg_hrv(hrv_merged[(hrv_merged["timestamp_min"] >= hrv_merged["sleep_end_min"]) & (hrv_merged["timestamp_min"] < hrv_merged["sleep_end_min"] + 30)][hrv.columns])
# evening: 30min before sleep -> wake_day. If sleep_start < 30min (early morning), span days: prev [start_prev,1440) + same [0,sleep_start_min]
sb_ev = sleep_bounds[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"})
hrv_ev_prev = hrv.copy()
hrv_ev_prev["wake_day"] = hrv_ev_prev["day_in_study"] + 1
hrv_ev_prev = hrv_ev_prev.merge(sb_ev, on=["id", "study_interval", "wake_day"], how="inner")
start_prev = (hrv_ev_prev["sleep_start_min"] - 30 + 1440) % 1440
mask_prev = (hrv_ev_prev["sleep_start_min"] >= 30) & (hrv_ev_prev["timestamp_min"] >= hrv_ev_prev["sleep_start_min"] - 30) & (hrv_ev_prev["timestamp_min"] < hrv_ev_prev["sleep_start_min"])
mask_prev = mask_prev | ((hrv_ev_prev["sleep_start_min"] < 30) & (hrv_ev_prev["timestamp_min"] >= start_prev))
hrv_ev_prev = hrv_ev_prev.loc[mask_prev].copy()
hrv_ev_prev["day_in_study"] = hrv_ev_prev["wake_day"]
hrv_ev_prev = hrv_ev_prev[hrv.columns]
hrv_ev_same = hrv.merge(sleep_bounds[["id", "study_interval", "day_in_study", "sleep_start_min"]], on=["id", "study_interval", "day_in_study"], how="inner")
hrv_ev_same = hrv_ev_same[(hrv_ev_same["sleep_start_min"] < 30) & (hrv_ev_same["timestamp_min"] < hrv_ev_same["sleep_start_min"])][hrv.columns]
hrv_evening = agg_hrv(pd.concat([hrv_ev_prev, hrv_ev_same], ignore_index=True))
print("HRV daily: full", len(hrv_full), "sleep", len(hrv_sleep), "morning", len(hrv_morning), "evening", len(hrv_evening))

HRV daily: full 4175 sleep 4042 morning 10 evening 1655


In [67]:
def agg_hr_window(path, window, sleep_bounds_df, cycle_days_df, ts_to_min_fn, key=None):
    key = key or ["id", "study_interval", "day_in_study"]
    sb = sleep_bounds_df
    agg_list = []
    for chunk in pd.read_csv(path, chunksize=500_000):
        chunk = chunk[(chunk["bpm"] >= 30) & (chunk["bpm"] <= 220) & (chunk["confidence"] >= 0.5)]
        chunk["timestamp_min"] = chunk["timestamp"].map(ts_to_min_fn)
        chunk = chunk.merge(cycle_days_df, on=key, how="inner")
        if window == "full":
            use = chunk[key + ["bpm"]]
        else:
            chunk = chunk.merge(sb, on=key, how="left")
            if window == "sleep":
                same = chunk[chunk["timestamp_min"] <= chunk["sleep_end_min"]][key + ["bpm"]]
                prev = chunk.drop(columns=["sleep_start_min", "sleep_end_min"], errors="ignore").copy()
                prev["wake_day"] = prev["day_in_study"] + 1
                prev = prev.merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"}), on=["id", "study_interval", "wake_day"], how="inner")
                prev = prev[prev["timestamp_min"] >= prev["sleep_start_min"]].copy()
                prev["day_in_study"] = prev["wake_day"]
                prev = prev[key + ["bpm"]]
                use = pd.concat([same, prev], ignore_index=True)
            elif window == "morning":
                use = chunk[(chunk["timestamp_min"] >= chunk["sleep_end_min"]) & (chunk["timestamp_min"] < chunk["sleep_end_min"] + 30)][key + ["bpm"]]
            elif window == "evening":
                # Prev day: need this night's sleep_start (attr to wake_day); same-day early: need this night's sleep_start (attr to day_in_study)
                # Drop sleep_* from first merge to avoid _x/_y duplicate cols in second merge
                chunk = chunk.drop(columns=["sleep_start_min", "sleep_end_min"], errors="ignore")
                chunk["wake_day"] = chunk["day_in_study"] + 1
                chunk = chunk.merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"}), on=["id", "study_interval", "wake_day"], how="inner")
                start_prev = (chunk["sleep_start_min"] - 30 + 1440) % 1440
                m_prev = (chunk["day_in_study"] == chunk["wake_day"] - 1) & (((chunk["sleep_start_min"] >= 30) & (chunk["timestamp_min"] >= chunk["sleep_start_min"] - 30) & (chunk["timestamp_min"] < chunk["sleep_start_min"])) | ((chunk["sleep_start_min"] < 30) & (chunk["timestamp_min"] >= start_prev)))
                use_prev = chunk.loc[m_prev, key + ["bpm"]].copy()
                use_prev["day_in_study"] = chunk.loc[m_prev, "wake_day"].values
                same_df = chunk[key + ["timestamp_min", "bpm"]].merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]], on=key, how="inner")
                m_same = (same_df["sleep_start_min"] < 30) & (same_df["timestamp_min"] < same_df["sleep_start_min"])
                use_same = same_df.loc[m_same, key + ["bpm"]].copy()
                use = pd.concat([use_prev, use_same], ignore_index=True)
            else:
                use = chunk[key + ["bpm"]]
        if len(use) == 0:
            continue
        agg_list.append(use.groupby(key, as_index=False)["bpm"].agg(hr_mean="mean", hr_std="std", hr_min="min", hr_max="max"))
    if not agg_list:
        return pd.DataFrame(columns=key + ["hr_mean", "hr_std", "hr_min", "hr_max"])
    return pd.concat(agg_list).groupby(key, as_index=False).agg(hr_mean=("hr_mean", "mean"), hr_std=("hr_std", "mean"), hr_min=("hr_min", "min"), hr_max=("hr_max", "max"))

In [68]:
def agg_wt_window(path, window, sleep_bounds_df, cycle_days_df, ts_to_min_fn, key=None):
    key = key or ["id", "study_interval", "day_in_study"]
    sb = sleep_bounds_df
    col = "temperature_diff_from_baseline"
    agg_list = []
    for chunk in pd.read_csv(path, chunksize=500_000):
        chunk = chunk[chunk[col].abs() <= 5]
        chunk["timestamp_min"] = chunk["timestamp"].map(ts_to_min_fn)
        chunk = chunk.merge(cycle_days_df, on=key, how="inner")
        if window == "full":
            use = chunk[key + [col]]
        else:
            chunk = chunk.merge(sb, on=key, how="left")
            if window == "sleep":
                same = chunk[chunk["timestamp_min"] <= chunk["sleep_end_min"]][key + [col]]
                prev = chunk.drop(columns=["sleep_start_min", "sleep_end_min"], errors="ignore").copy()
                prev["wake_day"] = prev["day_in_study"] + 1
                prev = prev.merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"}), on=["id", "study_interval", "wake_day"], how="inner")
                prev = prev[prev["timestamp_min"] >= prev["sleep_start_min"]].copy()
                prev["day_in_study"] = prev["wake_day"]
                prev = prev[key + [col]]
                use = pd.concat([same, prev], ignore_index=True)
            elif window == "morning":
                use = chunk[(chunk["timestamp_min"] >= chunk["sleep_end_min"]) & (chunk["timestamp_min"] < chunk["sleep_end_min"] + 30)][key + [col]]
            elif window == "evening":
                chunk = chunk.drop(columns=["sleep_start_min", "sleep_end_min"], errors="ignore")
                chunk["wake_day"] = chunk["day_in_study"] + 1
                chunk = chunk.merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"}), on=["id", "study_interval", "wake_day"], how="inner")
                start_prev = (chunk["sleep_start_min"] - 30 + 1440) % 1440
                m_prev = (chunk["day_in_study"] == chunk["wake_day"] - 1) & (((chunk["sleep_start_min"] >= 30) & (chunk["timestamp_min"] >= chunk["sleep_start_min"] - 30) & (chunk["timestamp_min"] < chunk["sleep_start_min"])) | ((chunk["sleep_start_min"] < 30) & (chunk["timestamp_min"] >= start_prev)))
                use_prev = chunk.loc[m_prev, key + [col]].copy()
                use_prev["day_in_study"] = chunk.loc[m_prev, "wake_day"].values
                same_df = chunk[key + ["timestamp_min", col]].merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]], on=key, how="inner")
                m_same = (same_df["sleep_start_min"] < 30) & (same_df["timestamp_min"] < same_df["sleep_start_min"])
                use_same = same_df.loc[m_same, key + [col]].copy()
                use = pd.concat([use_prev, use_same], ignore_index=True)
            else:
                use = chunk[key + [col]]
        if len(use) == 0:
            continue
        agg_list.append(use.groupby(key, as_index=False)[col].agg(wt_mean="mean", wt_std="std", wt_min="min", wt_max="max"))
    if not agg_list:
        return pd.DataFrame(columns=key + ["wt_mean", "wt_std", "wt_min", "wt_max"])
    return pd.concat(agg_list).groupby(key, as_index=False).agg(wt_mean=("wt_mean", "mean"), wt_std=("wt_std", "mean"), wt_min=("wt_min", "min"), wt_max=("wt_max", "max"))

In [69]:
# Daily: nightly_temperature (by day_in_study), resting_hr (aggregate by key, one row per day)
ct_daily = ct_night[["id", "study_interval", "sleep_end_day_in_study", "nightly_temperature"]].rename(
    columns={"sleep_end_day_in_study": "day_in_study"}
)
rhr = pd.read_csv(SUB / "resting_heart_rate_cycle.csv")[["id", "study_interval", "day_in_study", "value"]].rename(columns={"value": "resting_hr"})
rhr = rhr.groupby(["id", "study_interval", "day_in_study"], as_index=False)["resting_hr"].mean()
hrv_by_w = {"full": hrv_full, "sleep": hrv_sleep, "morning": hrv_morning, "evening": hrv_evening}
key = ["id", "study_interval", "day_in_study"]

In [70]:
# STEP 3+4: Aggregate HR/WT by window, merge with HRV, nightly_temperature, resting_hr; cycle_days as base
hr_path = SUB / "heart_rate_cycle.csv"
wt_path = SUB / "wrist_temperature_cycle.csv"
daily_by_window = {}
for w in ["full", "sleep", "morning", "evening"]:
    hr_w = agg_hr_window(hr_path, w, sleep_bounds, cycle_days, ts_to_min) if hr_path.exists() else pd.DataFrame(columns=key + ["hr_mean", "hr_std", "hr_min", "hr_max"])
    wt_w = agg_wt_window(wt_path, w, sleep_bounds, cycle_days, ts_to_min) if wt_path.exists() else pd.DataFrame(columns=key + ["wt_mean", "wt_std", "wt_min", "wt_max"])
    base = cycle_days.merge(hrv_by_w[w], on=key, how="left").merge(hr_w, on=key, how="left").merge(wt_w, on=key, how="left").merge(ct_daily, on=key, how="left").merge(rhr, on=key, how="left")
    daily_by_window[w] = base
    print(w, "rows", len(base))

# ── P1: 周期位置特征 (day_in_cycle, day_in_cycle_frac) ──────────────────────────
# day_in_cycle      = 本周期内天序号（0-indexed），如"今天是第 12 天"，不含总长信息
# day_in_cycle_frac = day_in_cycle / 28.0（以标准 28 天先验归一化，与实际周期长度无关）
#
# ⚠️ 禁止使用 day_in_cycle_norm = day_in_cycle / actual_cycle_len：
#    actual_cycle_len = days_until_next_menses + day_in_cycle，
#    即 day_in_cycle_norm 直接编码了预测目标 days_until_next_menses，构成数据泄露。
CYCLE_LEN_PRIOR = 28.0  # 临床标准先验周期长度，仅用于归一化

for w in daily_by_window:
    df = daily_by_window[w].sort_values(["id", "study_interval", "small_group_key", "day_in_study"])
    cycle_start = df.groupby(["id", "study_interval", "small_group_key"])["day_in_study"].transform("min")
    df["day_in_cycle"]      = (df["day_in_study"] - cycle_start).astype(np.float32)
    df["day_in_cycle_frac"] = (df["day_in_cycle"] / CYCLE_LEN_PRIOR).astype(np.float32)
    daily_by_window[w] = df
print("[P1] day_in_cycle + day_in_cycle_frac added (fixed 28-day prior, NO leakage).")
print("  sample:", daily_by_window["sleep"][["day_in_cycle","day_in_cycle_frac"]].describe().loc[["min","max","mean"]].round(2))

full rows 4825
sleep rows 4825
morning rows 4825
evening rows 4825
[P1] day_in_cycle + day_in_cycle_frac added (fixed 28-day prior, NO leakage).
  sample:       day_in_cycle  day_in_cycle_frac
min           0.00               0.00
max          81.00               2.89
mean         15.12               0.54


In [71]:
# ── P5 STEP 3.5: 腕温/夜间温度双相转折衍生特征 ──────────────────────────────────
# wt_shift_7v3  = 近3天 wt_mean 均值 - 前第3–9天 wt_mean 均值
#                  正值 ≥ 0.2 对应 Apple 算法的双相转折检测阈值（排卵后温度持续上升）
# temp_shift_7v3 = 同理对 nightly_temperature 计算
# 在缺失填充和 z 归一化之前计算（保留原始 °C 单位，信号含义直观）
SHIFT_SHORT = 3  # 近期窗口
SHIFT_LONG  = 7  # 参考窗口（近 3–9 天）

def add_biphasic_shift(df, col, out_col, short=3, long=7):
    df = df.sort_values(["id", "study_interval", "small_group_key", "day_in_study"])
    def _shift(x):
        recent = x.rolling(short, min_periods=1).mean()
        prior  = x.rolling(long, min_periods=max(1, long//2)).mean().shift(short)
        return (recent - prior).astype(np.float32)
    df[out_col] = df.groupby(["id", "study_interval"])[col].transform(_shift)
    return df

for w in daily_by_window:
    df = daily_by_window[w]
    if "wt_mean" in df.columns:
        df = add_biphasic_shift(df, "wt_mean",          "wt_shift_7v3",   SHIFT_SHORT, SHIFT_LONG)
    if "nightly_temperature" in df.columns:
        df = add_biphasic_shift(df, "nightly_temperature", "temp_shift_7v3", SHIFT_SHORT, SHIFT_LONG)
    daily_by_window[w] = df
print("[P5] Biphasic shift features added (wt_shift_7v3, temp_shift_7v3)")

# STEP 5: Missing value handling
# P3: 改为邻日线性插值（limit=3 天）替代全局均值，避免跨相位污染（Hamidovic 2023 显示各相位间差异显著）
# 超过 3 天连续缺失则填 0；同时保留 *_missing 指示列供模型参考
feat_cols_raw = [c for c in daily_by_window["full"].columns
                 if c not in key + ["small_group_key",
                                    "day_in_cycle", "day_in_cycle_norm", "day_in_cycle_frac",
                                    "wt_shift_7v3", "temp_shift_7v3",
                                    "hist_cycle_len_mean", "hist_cycle_len_std",
                                    "days_remaining_prior", "days_remaining_prior_log"]
                 and not c.endswith("_missing")
                 and pd.api.types.is_numeric_dtype(daily_by_window["full"][c])]
feat_cols = feat_cols_raw  # 用于后续 norm_cols 计算

INTERP_LIMIT = 3  # 最多插值补 3 天连续缺失；超过则填 0

for w, df in daily_by_window.items():
    out = df.sort_values(["id", "study_interval", "day_in_study"]).copy()
    for col in feat_cols:
        if col not in out.columns:
            continue
        miss = out[col].isna()
        out[col + "_missing"] = miss.astype(int)
        # 邻日线性插值（限于同一被试的观测期，不外推）
        out[col] = (
            out.groupby(["id", "study_interval"])[col]
            .transform(lambda x: x.interpolate(
                method="linear", limit=INTERP_LIMIT, limit_direction="both"
            ))
        )
        # 插值后仍为 NaN（连续缺失>3天或首末端无值）则填 0
        out.loc[out[col].isna(), col] = 0
    daily_by_window[w] = out
print("[P3] Missing fill: linear interp (limit=3) + 0; feat_cols:", feat_cols)

[P5] Biphasic shift features added (wt_shift_7v3, temp_shift_7v3)


[P3] Missing fill: linear interp (limit=3) + 0; feat_cols: ['rmssd_mean', 'lf_mean', 'hf_mean', 'lf_hf_ratio', 'hr_mean', 'hr_std', 'hr_min', 'hr_max', 'wt_mean', 'wt_std', 'wt_min', 'wt_max', 'nightly_temperature', 'resting_hr']


In [72]:
# ── P6: 历史周期长度先验特征 ────────────────────────────────────────────────────
# 核心思路（Wang 2025 方法的关键）：用「被试过去的周期长度」预测当前周期何时结束。
# hist_cycle_len_mean: 该被试当前周期「之前」所有周期的平均长度（历史均值）
# hist_cycle_len_std:  历史周期长度的标准差（个体内不确定性）
# 第一个周期（无历史数据）使用群体先验（约25天、标准差5天）
# 这两列在整个周期内保持恒定 → 不纳入 z 归一化，直接作为原值存入 CSV
POPULATION_CYCLE_MEAN = 25.0
POPULATION_CYCLE_STD  = 5.0

# Step 1: 计算每个周期的实际天数及起始日
cycle_info = (
    cycle_days
    .groupby(["id", "study_interval", "small_group_key"])
    .agg(
        cycle_start=("day_in_study", "min"),
        cycle_len=("day_in_study", "count")
    )
    .reset_index()
    .sort_values(["id", "cycle_start"])
    .reset_index(drop=True)
)

# Step 2: 按被试计算历史均值/标准差（严格只用当前周期之前的历史）
hist_means, hist_stds = [], []
for _, subj_df in cycle_info.groupby("id"):
    lens = subj_df["cycle_len"].values
    for i in range(len(lens)):
        if i == 0:
            hist_means.append(POPULATION_CYCLE_MEAN)
            hist_stds.append(POPULATION_CYCLE_STD)
        else:
            prev = lens[:i]
            hist_means.append(float(np.mean(prev)))
            hist_stds.append(float(np.std(prev)) if len(prev) > 1 else POPULATION_CYCLE_STD)

cycle_info["hist_cycle_len_mean"] = hist_means
cycle_info["hist_cycle_len_std"]  = hist_stds

# Step 3: 合并回每日数据，并立即计算先验剩余天数特征
for w in daily_by_window:
    daily_by_window[w] = daily_by_window[w].merge(
        cycle_info[["id", "study_interval", "small_group_key",
                    "hist_cycle_len_mean", "hist_cycle_len_std"]],
        on=["id", "study_interval", "small_group_key"],
        how="left"
    )
    daily_by_window[w]["hist_cycle_len_mean"] = (
        daily_by_window[w]["hist_cycle_len_mean"].fillna(POPULATION_CYCLE_MEAN).astype(np.float32)
    )
    daily_by_window[w]["hist_cycle_len_std"] = (
        daily_by_window[w]["hist_cycle_len_std"].fillna(POPULATION_CYCLE_STD).astype(np.float32)
    )

    # ── days_remaining_prior (核心特征) ─────────────────────────────────────────
    # 先验剩余天数 = 历史均值 - 当前周期内天序号（有符号）
    #   正值：周期尚未结束（还剩 X 天）
    #   负值：周期已超预期（超期 X 天），月经即将来临
    # days_remaining_prior_log：与目标 log1p(days_until_menses) 同量纲，
    #   模型只需在此基础上学习一个小修正，无需从零开始预测
    dpr = (daily_by_window[w]["hist_cycle_len_mean"]
           - daily_by_window[w]["day_in_cycle"]).astype(np.float64)
    daily_by_window[w]["days_remaining_prior"] = dpr.astype(np.float32)
    daily_by_window[w]["days_remaining_prior_log"] = (
        np.sign(dpr) * np.log1p(np.abs(dpr))
    ).astype(np.float32)

print("[P6] hist_cycle_len + days_remaining_prior features added")
print("  cycle_len range:", cycle_info["cycle_len"].min(), "–", cycle_info["cycle_len"].max(),
      "| hist_mean range:", round(cycle_info.iloc[1:]["hist_cycle_len_mean"].min(), 1),
      "–", round(cycle_info["hist_cycle_len_mean"].max(), 1))
sample = daily_by_window["sleep"][["day_in_cycle", "hist_cycle_len_mean",
                                    "days_remaining_prior", "days_remaining_prior_log"]]
print("  sample (sleep, first 3 rows of a cycle):\n",
      sample.groupby(level=0).head(3).head(9).to_string())


[P6] hist_cycle_len + days_remaining_prior features added
  cycle_len range: 2 – 82 | hist_mean range: 19.0 – 42.0
  sample (sleep, first 3 rows of a cycle):
    day_in_cycle  hist_cycle_len_mean  days_remaining_prior  days_remaining_prior_log
0           0.0                 25.0                  25.0                  3.258096
1           1.0                 25.0                  24.0                  3.218876
2           2.0                 25.0                  23.0                  3.178054
3           3.0                 25.0                  22.0                  3.135494
4           4.0                 25.0                  21.0                  3.091043
5           5.0                 25.0                  20.0                  3.044523
6           6.0                 25.0                  19.0                  2.995732
7           7.0                 25.0                  18.0                  2.944439
8           8.0                 25.0                  17.0                  

In [73]:
# STEP 6: Rolling z-normalisation（P4 改进版）
# 策略：优先使用「当前周期内前 K_CYCLE_EARLY 天（月经期）」作为基线（与 Wang 2025/Hamidovic 2023 对齐）
# 备用：若周期早期数据缺失 >60%，则回退到 K_LONG=28 天的滚动窗口历史基线
# 不论哪种方式，方差过小则 z=0，并裁剪 ±Z_CLIP
K_CYCLE_EARLY = 5   # 周期内前 N 天（月经期）作为周期内基线
K_LONG        = 28  # 长期滚动窗口（备用）
EPS    = 1e-6
Z_CLIP = 5.0
norm_cols = [c for c in feat_cols if not c.endswith("_missing")]

def _safe_z(val, mu, sig, eps, z_clip):
    if np.isnan(sig) or sig < eps:
        return 0.0
    return float(np.clip((val - mu) / sig, -z_clip, z_clip))

def rolling_z_v2(df, cols, k_early, k_long, eps, z_clip):
    """
    For each day t in cycle C:
      1. Collect the first k_early days of cycle C as early_baseline.
         If they contain enough non-NaN values (coverage ≥ 40%), use their mean/std.
      2. Otherwise, fall back to the rolling window of k_long days before day t
         (same logic as original rolling_z).
    """
    out = df.sort_values(["id", "study_interval", "small_group_key", "day_in_study"]).copy()
    for col in cols:
        if col not in out.columns:
            continue
        z = np.zeros(len(out), dtype=np.float32)
        for (iid, sid, sgk), grp in out.groupby(["id", "study_interval", "small_group_key"]):
            x       = grp[col].values
            idx     = grp.index
            n       = len(x)
            # Pre-compute early-cycle baseline from first k_early rows of this cycle
            early_vals = x[:k_early]
            valid_early = early_vals[~np.isnan(early_vals)]
            use_early = len(valid_early) >= max(1, int(k_early * 0.4))
            if use_early:
                mu_early  = np.nanmean(valid_early)
                sig_early = np.nanstd(valid_early)
            # Also keep full history for fallback (need values across all days in study_interval)
            # We approximate fallback using the per-study-interval position
            full_study = out[(out["id"] == iid) & (out["study_interval"] == sid)].sort_values("day_in_study")
            x_full = full_study[col].values
            idx_full = full_study.index.tolist()
            for i in range(n):
                global_i = idx_full.index(idx[i])  # position in full study timeline
                if i < k_early and use_early:
                    # During the early-cycle window itself: no baseline yet (not yet complete), use 0
                    z[idx[i]] = 0.0
                    continue
                if use_early:
                    z[idx[i]] = _safe_z(x[i], mu_early, sig_early, eps, z_clip)
                else:
                    # Fallback: rolling k_long in study timeline
                    hist = x_full[max(0, global_i - k_long): global_i]
                    if len(hist) == 0:
                        z[idx[i]] = 0.0
                    else:
                        mu_r  = np.nanmean(hist)
                        sig_r = np.nanstd(hist)
                        z[idx[i]] = _safe_z(x[i], mu_r, sig_r, eps, z_clip)
        out[col + "_z"] = z
    return out

for w in daily_by_window:
    daily_by_window[w] = rolling_z_v2(daily_by_window[w], norm_cols, K_CYCLE_EARLY, K_LONG, EPS, Z_CLIP)
print("[P4] Cycle-early baseline z done (K_early={}, K_long={})".format(K_CYCLE_EARLY, K_LONG))

[P4] Cycle-early baseline z done (K_early=5, K_long=28)


**Z 分数爆炸原因（已修复）**：当过去 K 天窗口内方差为 0（如 resting_hr 多天为 0、某特征多天常数）时，原逻辑用 `sig = eps(1e-6)`，导致 `z = (x - μ) / 1e-6` 极大（例：resting_hr 从 0 到 79 → z≈79e6）。修复：方差 < eps 时置 z=0；否则计算 z 后裁剪到 ±5。

In [74]:
# STEP 7: Output 4-window daily feature tables + index.csv
cycle_days.to_csv(OUT / "index.csv", index=False)
for w, name in [("full", "full"), ("sleep", "sleep"), ("morning", "morning"), ("evening", "evening")]:
    daily_by_window[w].to_csv(OUT / f"{name}.csv", index=False)
print("saved:", list(OUT.glob("*.csv")))

saved: [PosixPath('/Users/xujing/FYP/main_workspace/processed_data/2/full.csv'), PosixPath('/Users/xujing/FYP/main_workspace/processed_data/2/evening.csv'), PosixPath('/Users/xujing/FYP/main_workspace/processed_data/2/morning.csv'), PosixPath('/Users/xujing/FYP/main_workspace/processed_data/2/sleep.csv'), PosixPath('/Users/xujing/FYP/main_workspace/processed_data/2/index.csv')]


In [75]:
# Check: len(X) == len(index); model input uses _z columns
idx = pd.read_csv(OUT / "index.csv")
full_df = pd.read_csv(OUT / "full.csv")
z_cols = [c for c in full_df.columns if c.endswith("_z")]
X = full_df[z_cols].values if z_cols else full_df[[c for c in full_df.columns if c not in key and not c.endswith("_missing")]].values
print("index rows:", len(idx), "X shape:", X.shape, "len(X)==len(index):", len(X) == len(idx))



index rows: 4825 X shape: (4825, 14) len(X)==len(index): True
